In [0]:
from pyspark.sql import functions as f,Window
import re
from datetime import datetime
from pyspark.sql.types import *

# access legacy tables
cust = spark.read.table('madstagescd2.madstage.silver_customers')
ord = spark.read.table('madstagescd2.madstage.silver_orders')
prod = spark.read.table('madstagescd2.madstage.silver_products')

# order_id|customer_id|product_id|quantity|order_amount|order_date|   status|is_current|updated_dt|created_dt|
#|product_id|       product_name|   category|    brand|price|status|is_current|updated_dt|created_dt
#customer_id|customer_name|     city|customer_segment|           email|     phone|is_current|updated_dt|created_dt

cust_current = (
    cust
    .filter(f.col("is_current") == "Y")
)

prod_current = (
    prod
    .filter(f.col("is_current") == "Y")
)

gold = (
    ord.alias("o")
    .join(
        cust_current.alias("c"),
        on="customer_id",
        how="left"
    )
    .join(
        prod_current.alias("p"),
        on="product_id",
        how="left"
    )
    .select(
        "o.*",
        "p.product_name",
        "p.category",
        "p.brand",
        "p.price",
        "c.customer_name",
        "c.city",
        "c.customer_segment",
        "c.email",
        "c.phone"
    )
)

gold.createOrReplaceTempView('gold')

spark.sql("""
         merge into madstagescd2.madstage.goldcustomersorders as t 
         using gold as s 
         on t.order_id = s.order_id
         when matched then 
         update set 
         t.order_id = s.order_id,
         t.customer_id = s.customer_id,
         t.product_id = s.product_id,
         t.quantity = s.quantity,
         t.order_amount = s.order_amount,
         t.order_date = s.order_date,
         t.status = s.status,
         t.is_current = s.is_current,
         t.updated_dt = s.updated_dt,
         t.product_name = s.product_name,
         t.category = s.category,
         t.brand = s.brand,
         t.price = s.price,
         t.customer_name = s.customer_name,
         t.city = s.city,
         t.customer_segment = s.customer_segment,
         t.email = s.email,
         t.phone = s.phone
         when not matched then
         insert *   
""")



